# 🎯 BookVoice-AI — CosyVoice2-0.5B Türkisch
**Neuer Account · Drive Speicher · Einmal installieren, immer nutzen.**

In [ ]:
#@title 💾 Schritt 0: Google Drive verbinden (einmalig)
from google.colab import drive
drive.mount('/content/drive')

import os
# Modell in Drive speichern → nie wieder neu laden!
os.environ['MODELSCOPE_CACHE'] = '/content/drive/MyDrive/cosyvoice_models'
print('✅ Drive verbunden! Modelle werden in Drive gespeichert.')

In [ ]:
#@title ⚙️ Schritt 1: Installation + Modell laden (~15 Min beim ersten Mal)
import sys, os

# CosyVoice klonen
if not os.path.exists('/content/CosyVoice'):
    print('CosyVoice klonen...')
    os.system('git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git')

sys.path.insert(0, '/content/CosyVoice')
sys.path.insert(0, '/content/CosyVoice/third_party/Matcha-TTS')

# requirements.txt zuerst (Basis)
print('Pakete installieren...')
os.system('pip install -q -r /content/CosyVoice/requirements.txt')
# Fehlende Pakete für Python 3.12 + Colab
os.system('pip install -q hyperpyyaml modelscope onnxruntime conformer wget openai-whisper hydra-core')
print('Pakete OK!')

# PyTorch fix — NUR EINMAL
import torch
if not hasattr(torch, '_load_patched'):
    _orig = torch.load
    torch.load = lambda *a, **kw: _orig(*a, **{**kw, 'weights_only': False})
    torch._load_patched = True

# Modell laden
print('Modell laden (CosyVoice2-0.5B)...')
from cosyvoice.cli.cosyvoice import CosyVoice2
model = CosyVoice2('iic/CosyVoice2-0.5B', load_jit=False, load_trt=False)
print('✅ FERTIG! Jetzt Schritt 2 ausführen.')

In [ ]:
#@title 🎤 Schritt 2: Stimme hochladen + Türkisch generieren
from google.colab import files
import torchaudio, whisper, sys, torch
from IPython.display import Audio, display

# Stimme hochladen
print('Stimme hochladen (WAV/MP3):')
uploaded = files.upload()
voice_file = list(uploaded.keys())[0]

# Stimme laden - Mono - 8 Sekunden
speech, sr = torchaudio.load(voice_file)
if sr != 16000:
    speech = torchaudio.functional.resample(speech, sr, 16000)
if speech.shape[0] == 2:
    speech = speech.mean(dim=0, keepdim=True)
speech_8s = speech[:, :16000*8]
torchaudio.save('/content/stimme_8s.wav', speech_8s, 16000)
print(f'✅ Stimme: {speech_8s.shape[1]/16000:.1f} Sek, Mono, 16kHz')

# Transkription
print('Transkribiere...')
w = whisper.load_model('tiny')
result = w.transcribe('/content/stimme_8s.wav', language='tr')
prompt_text = result['text'].strip()
print(f'Prompt Text: {prompt_text}')

# Generieren mit Sprach-Tag für Türkisch
ziel_text = 'Merhaba, benim adım Ahrar. Ben bir Türk yazarıyım. Kitaplarımda Osmanlı tarihini anlatıyorum.' #@param {type:"string"}
print('Generiere Türkisch...')
for i, res in enumerate(model.inference_zero_shot(
    '<|tr|>' + ziel_text,
    prompt_text,
    speech_8s,
    stream=False
)):
    torchaudio.save('/content/cosy_output.wav', res['tts_speech'], model.sample_rate)
    print('✅ Fertig! Anhören:')
    display(Audio('/content/cosy_output.wav'))

In [ ]:
#@title ⏳ Schritt 3: Session aktiv halten — IMMER laufen lassen!
import time
print('Session läuft... (nicht stoppen!)')
while True:
    time.sleep(60)
    print('.', end='', flush=True)